## На семинаре вам предстоит выполнить задания ноутбука и улучшить скор.

In [1]:
!unzip -q -o ./data/sirius-nlp-2026-language-proficiency-classification.zip -d ./data/

In [1]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [2]:
import numpy as np
import pandas as pd

from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [3]:
train = pd.read_csv('data/train.csv')

X = train.drop(columns=['label'])
y = train['label']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
tfidf = TfidfVectorizer(max_features=256, analyzer='word', ngram_range=(1, 2))

train_vectors = tfidf.fit_transform(X_train['text']).toarray()
val_vectors = tfidf.transform(X_val['text']).toarray()

train_vectors = pd.DataFrame(train_vectors, index=X_train.index)
val_vectors = pd.DataFrame(val_vectors, index=X_val.index)

In [5]:
clf = LogisticRegression(max_iter=500, random_state=42)
clf.fit(train_vectors, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,500
,multi_class,'deprecated'


In [6]:
val_preds = clf.predict(val_vectors)
print(f1_score(y_val, val_preds, average='macro'))

0.45718843410890614


In [7]:
test_df = pd.read_csv('data/test.csv')

test_vectors = tfidf.transform(test_df['text']).toarray()

preds = clf.predict(test_vectors)
test_df['label'] = preds.tolist()
test_df[['text', 'label']].to_csv('submission_0.csv', index=False)

## Задание 1

Добавьте:
- нормализацию текста
- char level модель векторизации, для обоих моделей возьмите побольше фичей
- используйте еruncated-SVD для формированяи векторов меньшей размерности
- кросс-валидацию, стратифицированную по лейблам и категориальным признакам

In [32]:
import re
from tqdm import tqdm
import string

from scipy.sparse import hstack
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedKFold

import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('names', quiet=True)


True

In [33]:
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')

In [34]:
train_df.head()

,text,label,language,topic
0,Language learning is fun and important. It hel...,A1,en,Language Learning
1,The Industrial Revolution was a big change. It...,A1,en,The Industrial Revolution
2,Traditional cuisine is food from a country or ...,A1,en,Traditional Cuisine
3,Street food is very popular in many cities. Pe...,A1,en,Street Food
4,The Renaissance was a time of change in Europe...,A1,en,The Renaissance


In [35]:
train_df['language'].value_counts()

language
pt    241
de    241
ru    240
fr    240
es    240
it    240
ar    240
zh    240
en    239
nl    239
Name: count, dtype: int64

### 1. Нормализация текста


In [36]:
train_df["label"] = train_df["label"].astype(str)
train_df["language"] = train_df["language"].astype(str)
train_df["topic"] = train_df["topic"].str.lower().astype(str)
train_df["text"] = train_df["text"].str.lower().astype(str)

test_df["language"] = test_df["language"].astype(str)
test_df["topic"] = test_df["topic"].str.lower().astype(str)
test_df["text"] = test_df["text"].str.lower().astype(str)

train_df.head()

,text,label,language,topic
0,language learning is fun and important. it hel...,A1,en,language learning
1,the industrial revolution was a big change. it...,A1,en,the industrial revolution
2,traditional cuisine is food from a country or ...,A1,en,traditional cuisine
3,street food is very popular in many cities. pe...,A1,en,street food
4,the renaissance was a time of change in europe...,A1,en,the renaissance


In [39]:
def default_tokenizer(text):
    text = word_tokenize(text)
    text = [char for char in text if char not in string.punctuation]
    return text

In [40]:
train_df['topic_processed'] = train_df['topic'].apply(default_tokenizer)
train_df.head()

,text,label,language,topic,topic_processed
0,language learning is fun and important. it hel...,A1,en,language learning,"[language, learning]"
1,the industrial revolution was a big change. it...,A1,en,the industrial revolution,"[the, industrial, revolution]"
2,traditional cuisine is food from a country or ...,A1,en,traditional cuisine,"[traditional, cuisine]"
3,street food is very popular in many cities. pe...,A1,en,street food,"[street, food]"
4,the renaissance was a time of change in europe...,A1,en,the renaissance,"[the, renaissance]"


In [1]:
import pandas as pd
import spacy
import re

# 1. Словарь соответствия языков и моделей SpaCy
# Нужно предварительно скачать эти модели: python -m spacy download <название>
nlp_models = {
    'en': spacy.load("en_core_web_sm"),
    'ru': spacy.load("ru_core_news_sm"),
    'de': spacy.load("de_core_news_sm"),
    'fr': spacy.load("fr_core_news_sm"),
    'es': spacy.load("es_core_news_sm"),
    'it': spacy.load("it_core_news_sm"),
    'pt': spacy.load("pt_core_news_sm"),
    'nl': spacy.load("nl_core_news_sm"),
    'zh': spacy.load("zh_core_web_sm"), # Для китайского нужен спец. пакет
    'ar': spacy.load("xx_ent_wiki_sm") # Для арабского в SpaCy часто используют мультиязычную модель или Stanza
}

def clean_basic(text):
    """Базовая очистка для всех языков"""
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # Удаление ссылок
    text = re.sub(r'<.*?>', '', text) # Удаление HTML
    return text.strip()

def normalize_text(text, lang):
    # Если языка нет в списке, используем английский или оставляем как есть
    if lang not in nlp_models:
        return clean_basic(text)
    
    doc = nlp_models[lang](clean_basic(text))
    
    # Лемматизация, удаление пунктуации и стоп-слов
    tokens = [
        token.lemma_ for token in doc 
        if not token.is_stop and not token.is_punct and not token.is_space
    ]
    
    # Для китайского токены обычно не соединяют пробелами при выводе, 
    # но для большинства задач NLP пробел полезен
    return " ".join(tokens)

# Пример использования
df = pd.DataFrame({
    'text': ["Привет мир", "Hello world", "Bonjour le monde"],
    'lang': ['ru', 'en', 'fr']
})

# Применяем нормализацию
df['cleaned_text'] = df.apply(lambda x: normalize_text(x['text'], x['lang']), axis=1)
print(df)

OSError: [E050] Can't find model 'en_core_web_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

### 2. char + word level TF-IDF модели

In [ ]:
# YOUR CODE

### 3. Понижение размерности с помощью truncated SVD

In [ ]:
# YOUR CODE

### 4. Cтратификация выборки по лейблу и языку

In [ ]:
# YOUR CODE

### 5. Полный пайплайн кросс-валидации с сбором предсказаний на тесте

In [ ]:
# YOUR CODE

### 6. Обучение модели на полном наборе данных

In [ ]:
# YOUR CODE

### 7. Сборка сабмита с предсказаниями всех моделей

In [88]:
index_to_label = {
    0: 'A1',
    1: 'A2',
    2: 'B1',
    3: 'B2',
    4: 'C1',
    5: 'C2'
}

In [ ]:
test_pred_avg = np.mean(test_preds_cv, axis=0)
final_preds = [index_to_label[pred] for pred in test_pred_avg.argmax(axis=1)]

test_df["label"] = final_preds
test_df[["text", "label"]].to_csv("submission_1.csv", index=False)

## Задание 2

Выполните:
- извлеките признаки из модели BERT (google-bert/bert-base-multilingual-cased)
- постройте классификатор исключительно на извлечённых признаках
- постройте классификатор на объеденённых признаках BERT с признаками из задания 1

In [11]:
import torch

from transformers import AutoTokenizer, AutoModel

In [6]:
train = pd.read_csv('/kaggle/input/competitions/vsosh-2026-nlp/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/vsosh-2026-nlp/test.csv')

### Инициализируем модель

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-multilingual-cased")
model = AutoModel.from_pretrained("google-bert/bert-base-multilingual-cased")
model.to(device)
model.eval()
None

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google-bert/bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### 1. Напишите функцию батчевого сбора эмбеддингов текстов

In [ ]:
def mean_pool(last_hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
    return (last_hidden * mask).sum(1) / mask.sum(1)


@torch.no_grad()
def embed_texts(texts, batch_size=32, max_len=256):
    all_embeddings = []

    # YOUR CODE

    return np.vstack(all_embeddings)

### 2. mBERT embeddings

In [9]:
train_bert = embed_texts(train["text"].tolist())

test_bert = embed_texts(test_df["text"].tolist())

100%|██████████| 19/19 [00:04<00:00,  4.21it/s]


### 3. TF-IDF + SVD embeddings

In [ ]:
# YOUR CODE

### 4. CV training function

In [ ]:
# YOUR CODE

### 5. Собираем предсказания моделей

In [ ]:
stratify_key = train["label"].astype(str) + "_" + train["language"].astype(str)

pred_bert = train_cv(X_bert, Y, T_bert, "mBERT", stratify_key)
pred_lsa = train_cv(X_tfidf, Y, T_tfidf, "TF-IDF + SVD", stratify_key)
pred_hybrid = train_cv(X_hybrid, Y, T_hybrid, "HYBRID", stratify_key)


Training: mBERT
fold 0 F1: 0.634
fold 1 F1: 0.655
fold 2 F1: 0.669
fold 3 F1: 0.674
fold 4 F1: 0.660
CV mean F1: 0.658

Training: TFIDF+SVD
fold 0 F1: 0.542
fold 1 F1: 0.571
fold 2 F1: 0.547
fold 3 F1: 0.544
fold 4 F1: 0.535
CV mean F1: 0.548

Training: HYBRID
fold 0 F1: 0.623
fold 1 F1: 0.658
fold 2 F1: 0.671
fold 3 F1: 0.670
fold 4 F1: 0.676
CV mean F1: 0.660


### 6. Собираем сабмит

In [35]:
index_to_label = {
    0: 'A1',
    1: 'A2',
    2: 'B1',
    3: 'B2',
    4: 'C1',
    5: 'C2'
}

In [36]:
bert_labels = [index_to_label[np.argmax(pred)] for pred in pred_bert]
lsa_labels = [index_to_label[np.argmax(pred)] for pred in pred_lsa]
hybrid_labels = [index_to_label[np.argmax(pred)] for pred in pred_hybrid]

test_df["label"] = bert_labels
test_df[["text", "label"]].to_csv("submission_2_0.csv", index=False)

test_df["label"] = lsa_labels
test_df[["text", "label"]].to_csv("submission_2_1.csv", index=False)

test_df["label"] = hybrid_labels
test_df[["text", "label"]].to_csv("submission_2_2.csv", index=False)


final_pred = pred_lsa * 0.1 + pred_bert * 0.4 + pred_hybrid + 0.5
labels = [index_to_label[np.argmax(pred)] for pred in final_pred]

test_df["label"] = labels
test_df[["text", "label"]].to_csv("submission_2_3.csv", index=False)

## Задание 3

Выполните:
- дообучение данной модели BERT на задачу классификации
- дообучение с заморозкой всех слоёв кроме последних 3х

In [1]:
from sklearn.utils.class_weight import compute_class_weight

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)


In [33]:
train_df = pd.read_csv('/kaggle/input/competitions/vsosh-2026-nlp/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/vsosh-2026-nlp/test.csv')

label_to_idx = {
    'A1': 0,
    'A2': 1,
    'B1': 2,
    'B2': 3,
    'C1': 4,
    'C2': 5
}

train_df['label'] = train_df['label'].apply(lambda x: label_to_idx[x])

num_labels = train_df["label"].nunique()

In [34]:
stratify_key = train_df["label"].astype(str) + "_" + train_df["language"].astype(str)

train_part, val_part = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=stratify_key
)

In [35]:
train_ds = Dataset.from_pandas(train_part[["text", "label"]])
val_ds = Dataset.from_pandas(val_part[["text", "label"]])
test_ds = Dataset.from_pandas(test_df[["text"]])

In [36]:
model_name = "google-bert/bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding=False,
        max_length=256
    )


train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

Map:   0%|          | 0/2160 [00:00<?, ? examples/s]

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

In [47]:
trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1
1,1.131275,0.615554,0.538781
2,0.604251,0.585115,0.617101
3,0.500229,0.518210,0.682564
4,0.438800,0.535381,0.706397


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=540, training_loss=0.6256765577528212, metrics={'train_runtime': 220.682, 'train_samples_per_second': 39.151, 'train_steps_per_second': 2.447, 'total_flos': 1116576529801920.0, 'train_loss': 0.6256765577528212, 'epoch': 4.0})

In [48]:
print(trainer.evaluate())

Best validation metrics:


{'eval_loss': 0.5353808999061584, 'eval_macro_f1': 0.7063974363202038, 'eval_runtime': 1.8165, 'eval_samples_per_second': 132.122, 'eval_steps_per_second': 4.404, 'epoch': 4.0}


In [51]:
index_to_label = {
    0: 'A1',
    1: 'A2',
    2: 'B1',
    3: 'B2',
    4: 'C1',
    5: 'C2'
}

In [52]:
test_pred = trainer.predict(test_ds)
preds = test_pred.predictions.argmax(axis=1)

labels = [index_to_label[pred] for pred in preds]

test_df["label"] = labels
test_df[["text", "label"]].to_csv("submission_3_0.csv", index=False)

### Эксперимент с заморозкой слоёв

In [53]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [55]:
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [ ]:
# YOUR CODE


print(f"Trainable params: {trainable}")
print(f"Total params: {total}")
print(f"Trainable %: {100 * trainable / total:.2f}%")

Trainable params: 21858822
Total params: 177858054
Trainable %: 12.29%


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1
1,1.366244,0.772234,0.550336
2,0.783270,0.627770,0.688057
3,0.590564,0.602025,0.723754
4,0.579687,0.575301,0.693783


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=540, training_loss=0.7794546056676794, metrics={'train_runtime': 116.4419, 'train_samples_per_second': 74.2, 'train_steps_per_second': 4.638, 'total_flos': 1116576529801920.0, 'train_loss': 0.7794546056676794, 'epoch': 4.0})

In [63]:
print(trainer.evaluate())

{'eval_loss': 0.6018356084823608, 'eval_macro_f1': 0.7237538290169869, 'eval_runtime': 1.8428, 'eval_samples_per_second': 130.239, 'eval_steps_per_second': 4.341, 'epoch': 4.0}


In [64]:
test_pred = trainer.predict(test_ds)
preds = test_pred.predictions.argmax(axis=1)

labels = [index_to_label[pred] for pred in preds]

test_df["label"] = labels
test_df[["text", "label"]].to_csv("submission_3_1.csv", index=False)